In [1]:
import sqlite3
import pandas as pd
import numpy as np
import pickle
import re
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
df_all = pd.read_pickle("../data/tags_alteration_full_2025-10_dm.pkl")
df = df_all[df_all['status'] != 'non-legit'].copy()
df = df[df['old_snap_timestamp'] >= pd.Timestamp('2016-02-23')]

print(f"Total alterations: {len(df_all):_}")
print(f"Alterations used (excluding non-legit): {len(df):_}")
print(f"Non-legit filtered out: {len(df_all) - len(df):_}")

Total alterations: 15_166_153
Alterations used (excluding non-legit): 10_171_046
Non-legit filtered out: 4_995_107


In [8]:
correlation = pd.read_csv("step2_altered_tag_matches.csv")

In [9]:
# Add prefix to correlation columns (except origin_url which is the merge key)
correlation_with_prefix = correlation.add_prefix('cor_')
correlation_with_prefix = correlation_with_prefix.rename(columns={'cor_origin_url': 'origin_url'})
correlation_with_prefix = correlation_with_prefix.rename(columns={'cor_rev': 'tag_name'})
# Merge with df on origin_url
df_merged = df.merge(correlation_with_prefix, on=['origin_url', 'tag_name'], how='left')

print(f"Rows in df: {len(df):_}")
print(f"Rows in correlation: {len(correlation):_}")
print(f"Rows after merge: {len(df_merged):_}")
print(f"Rows with correlation match: {df_merged['cor_altered_tag'].notna().sum():_}")

df_merged

Rows in df: 10_171_046
Rows in correlation: 35
Rows after merge: 10_171_053
Rows with correlation match: 41


,origin_url,tag_name,type,old_snapshot,old_snapshot_cpt,old_snap_timestamp,old_revision,old_rev_timestamp,old_root_dir,new_snapshot,new_snapshot_cpt,new_snap_timestamp,new_revision,new_rev_timestamp,new_root_dir,min_delta,snap_year,rev_year,category,delta,platform,stars,creation_type,creation_rev,creation_rev_ts,creation_root_dir,creation_snapshot,creation_snap_ts,creation_delta,deadtime,status,cor_attr_path,cor_pname,cor_version,cor_forge_type,cor_owner,cor_repo,cor_rev_type,cor_hash,cor_fetch_url,cor_altered_tag,cor_sha_before_alteration,cor_old_snap_timestamp,cor_sha_after_alteration,cor_alteration_detected_at
0,https://github.com/innovation-cat/innovation-cat.github.io,refs/tags/2.0.0,lightweight,swh:1:snp:d10c567a401e6a134a99899681bb2ee90007be0d,1,2022-03-05 08:11:53,swh:1:rev:29e7fce9fc88a55bdafb507f6506bd6f153260b0,2020-04-27 03:35:14,swh:1:dir:c612010c922cea11ecdfa384db474123a0225d73,swh:1:snp:bd6d353312a43153ac7a927924bcf04e3c77e771,2,2022-03-12 17:36:22,NaN,NaT,NaN,2022-03-05 08:11:53,2022,NaN,Deletion,0,GitHub,0.0,NaN,NaN,NaT,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://github.com/innovation-cat/innovation-cat.github.io,refs/tags/2.2.0,lightweight,swh:1:snp:d10c567a401e6a134a99899681bb2ee90007be0d,1,2022-03-05 08:11:53,swh:1:rev:16a7fefff1f1fb674d1ed0328099a8d6ac0d38bb,2020-04-27 19:31:38,swh:1:dir:3727b13bc24dad71024adebc98eb17a2b9310d35,swh:1:snp:bd6d353312a43153ac7a927924bcf04e3c77e771,2,2022-03-12 17:36:22,NaN,NaT,NaN,2022-03-05 08:11:53,2022,NaN,Deletion,0,GitHub,0.0,NaN,NaN,NaT,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://github.com/innovation-cat/innovation-cat.github.io,refs/tags/3.0.0,lightweight,swh:1:snp:d10c567a401e6a134a99899681bb2ee90007be0d,1,2022-03-05 08:11:53,swh:1:rev:85401c9f014ff7d8950b1930e732b73fd6d08043,2020-05-07 07:24:36,swh:1:dir:65029e9180c14f7e3c5d0d1d3cf0b9b697b5e60e,swh:1:snp:bd6d353312a43153ac7a927924bcf04e3c77e771,2,2022-03-12 17:36:22,NaN,NaT,NaN,2022-03-05 08:11:53,2022,NaN,Deletion,0,GitHub,0.0,NaN,NaN,NaT,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://github.com/innovation-cat/innovation-cat.github.io,refs/tags/2.3.0,lightweight,swh:1:snp:d10c567a401e6a134a99899681bb2ee90007be0d,1,2022-03-05 08:11:53,swh:1:rev:6fa41ccd9ece1292f0df3554e0c5f6680c82e818,2020-04-29 08:42:15,swh:1:dir:1edad0e47774c0fb05657616f6d8a4210cbcacb0,swh:1:snp:bd6d353312a43153ac7a927924bcf04e3c77e771,2,2022-03-12 17:36:22,NaN,NaT,NaN,2022-03-05 08:11:53,2022,NaN,Deletion,0,GitHub,0.0,NaN,NaN,NaT,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://github.com/innovation-cat/innovation-cat.github.io,refs/tags/4.0.0,lightweight,swh:1:snp:d10c567a401e6a134a99899681bb2ee90007be0d,1,2022-03-05 08:11:53,swh:1:rev:f2b6f5eb8c7579b7890fc706622c83fd1e89aace,2020-07-12 07:31:45,swh:1:dir:d4fa8f91bb1635172644c67d347da0733bfa90db,swh:1:snp:bd6d353312a43153ac7a927924bcf04e3c77e771,2,2022-03-12 17:36:22,NaN,NaT,NaN,2022-03-05 08:11:53,2022,NaN,Deletion,0,GitHub,0.0,NaN,NaN,NaT,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10171048,https://github.com/SolarFramework/Sample-Slam,refs/tags/SolARPipeline_SLAM/0.9.1/win,lightweight,swh:1:snp:4a34b88a1266ea49f7b1f0504f01d72de764ec8c,4,2021-07-17 13:36:17,swh:1:rev:48380bb57d9e84bead558a0e953f23779e149696,2021-07-09 09:28:09,swh:1:dir:56d915c61856d34ee4556d8b8035283a5d32282b,swh:1:snp:6f291f650d0a26a7825076fc7d1ed55bcfcb7965,5,2021-08-09 02:51:43,swh:1:rev:e97a40823a0b43d60a3e9d1e61be2a8c8c58039b,2021-07-21 07:07:12,swh:1:dir:0a14456619b66f1dc59fc6d52b82ba9b192f8499,2021-07-17 13:36:17,2021,2021.0,Move,0,GitHub,0.0,NaN,NaN,NaT,NaN,NaN,NaT,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [12]:
correlation_merge = df_merged[df_merged['cor_altered_tag'].notna()]

In [22]:
move_corr = correlation_merge[(correlation_merge['old_root_dir']!=correlation_merge['new_root_dir']) & (correlation_merge['old_root_dir']!=correlation_merge['creation_root_dir']) & ((correlation_merge['new_root_dir'].notna()) | (correlation_merge['creation_root_dir'].notna()))]
move_corr = move_corr.drop_duplicates()
print(f"tag lost: {len(move_corr):_}")

tag lost: 31


In [25]:
move_corr[move_corr['origin_url']=="https://github.com/geigi/cozy"]

,origin_url,tag_name,type,old_snapshot,old_snapshot_cpt,old_snap_timestamp,old_revision,old_rev_timestamp,old_root_dir,new_snapshot,new_snapshot_cpt,new_snap_timestamp,new_revision,new_rev_timestamp,new_root_dir,min_delta,snap_year,rev_year,category,delta,platform,stars,creation_type,creation_rev,creation_rev_ts,creation_root_dir,creation_snapshot,creation_snap_ts,creation_delta,deadtime,status,cor_attr_path,cor_pname,cor_version,cor_forge_type,cor_owner,cor_repo,cor_rev_type,cor_hash,cor_fetch_url,cor_altered_tag,cor_sha_before_alteration,cor_old_snap_timestamp,cor_sha_after_alteration,cor_alteration_detected_at
1950608,https://github.com/geigi/cozy,refs/tags/1.3.0,annotated,swh:1:snp:bfeb954cf866b15c9b5118bf333789f952a5d2c9,125,2023-12-04 13:28:03,swh:1:rev:1ba3745967949d50781cd1a430df982130c5437d,2023-12-03 14:47:09,swh:1:dir:781b33e7652d8d67e3ca4470770a3b7d6694f42f,swh:1:snp:56415de1de815c995b40b61e9218e434be5791ff,131,2024-02-17 06:52:31,NaN,NaT,NaN,2024-02-12 09:24:57,2024,NaN,Deletion,69,GitHub,955.0,lightweight,swh:1:rev:815821dba39f013738be6c2e70281ab52f77cae6,2024-02-16 14:05:41,swh:1:dir:a26bbb823bdfdd05900282c34b2f37a76f78f766,56415de1de815c995b40b61e9218e434be5791ff,2024-02-17 06:52:31,2024-02-17 06:52:31,0 days,legit,cozy,cozy,1.3.0,fetchFromGitHub,geigi,cozy,tag,sha256-oMgdz2dny0u1XV13aHu5s8/pcAz8z/SAOf4hbCDsdjw,https://github.com/geigi/cozy/archive/refs/tags/1.3.0.tar.gz,refs/tags/1.3.0,swh:1:snp:bfeb954cf866b15c9b5118bf333789f952a5d2c9,1.701696e+09,swh:1:snp:56415de1de815c995b40b61e9218e434be5791ff,1.708153e+09
1950609,https://github.com/geigi/cozy,refs/tags/1.3.0,annotated,swh:1:snp:bfeb954cf866b15c9b5118bf333789f952a5d2c9,125,2023-12-04 13:28:03,swh:1:rev:1ba3745967949d50781cd1a430df982130c5437d,2023-12-03 14:47:09,swh:1:dir:781b33e7652d8d67e3ca4470770a3b7d6694f42f,swh:1:snp:56415de1de815c995b40b61e9218e434be5791ff,131,2024-02-17 06:52:31,NaN,NaT,NaN,2024-02-12 09:24:57,2024,NaN,Deletion,69,GitHub,955.0,lightweight,swh:1:rev:815821dba39f013738be6c2e70281ab52f77cae6,2024-02-16 14:05:41,swh:1:dir:a26bbb823bdfdd05900282c34b2f37a76f78f766,56415de1de815c995b40b61e9218e434be5791ff,2024-02-17 06:52:31,2024-02-17 06:52:31,0 days,legit,cozy,cozy,1.3.0,fetchFromGitHub,geigi,cozy,tag,sha256-oMgdz2dny0u1XV13aHu5s8/pcAz8z/SAOf4hbCDsdjw,https://github.com/geigi/cozy/archive/refs/tags/1.3.0.tar.gz,refs/tags/1.3.0,swh:1:snp:56415de1de815c995b40b61e9218e434be5791ff,1.708153e+09,swh:1:snp:f4e81bc603947f1644e193f8edc92c3707659ab0,1.710721e+09
1950610,https://github.com/geigi/cozy,refs/tags/1.3.0,lightweight,swh:1:snp:56415de1de815c995b40b61e9218e434be5791ff,131,2024-02-17 06:52:31,swh:1:rev:815821dba39f013738be6c2e70281ab52f77cae6,2024-02-16 14:05:41,swh:1:dir:a26bbb823bdfdd05900282c34b2f37a76f78f766,swh:1:snp:f4e81bc603947f1644e193f8edc92c3707659ab0,133,2024-03-18 00:12:54,swh:1:rev:4c978ec53f0d86de116e4db471821af5138f8edc,2024-03-01 14:44:03,swh:1:dir:0361049db53403f4bc2d084f7774c6924b036561,2024-02-24 14:11:06,2024,2024.0,Move,7,GitHub,955.0,NaN,NaN,NaT,NaN,NaN,NaT,NaT,NaT,NaN,cozy,cozy,1.3.0,fetchFromGitHub,geigi,cozy,tag,sha256-oMgdz2dny0u1XV13aHu5s8/pcAz8z/SAOf4hbCDsdjw,https://github.com/geigi/cozy/archive/refs/tags/1.3.0.tar.gz,refs/tags/1.3.0,swh:1:snp:bfeb954cf866b15c9b5118bf333789f952a5d2c9,1.701696e+09,swh:1:snp:56415de1de815c995b40b61e9218e434be5791ff,1.708153e+09
1950611,https://github.com/geigi/cozy,refs/tags/1.3.0,lightweight,swh:1:snp:56415de1de815c995b40b61e9218e434be5791ff,131,2024-02-17 06:52:31,swh:1:rev:815821dba39f013738be6c2e70281ab52f77cae6,2024-02-16 14:05:41,swh:1:dir:a26bbb823bdfdd05900282c34b2f37a76f78f766,swh:1:snp:f4e81bc603947f1644e193f8edc92c3707659ab0,133,2024-03-18 00:12:54,swh:1:rev:4c978ec53f0d86de116e4db471821af5138f8edc,2024-03-01 14:44:03,swh:1:dir:0361049db53403f4bc2d084f7774c6924b036561,2024-02-24 14:11:06,2024,2024.0,Move,7,GitHub,955.0,NaN,NaN,NaT,NaN,NaN,NaT,NaT,NaT,NaN,cozy,cozy,1.3.0,fetchFromGitHub,geigi,cozy,tag,sha256-oMgdz2dny0u1XV13aHu5s8/pcAz8z/SAOf4hbCDsdjw,https://github.c